# isitfair hardening: `compas`

Step-by-step audit of the `compas` prediction table built by
`hardening/datasets/registry.py` (see `hardening/datasets/CARDS.md`). The
audited model is the logistic-regression column `y_proba_lr`; every stage is
its own cell, timed with `perf_counter`. Warnings and exceptions are recorded
per stage in `RECORD` and written to `hardening/notebooks/results/`.

Environment knobs: `HARDEN_B` (bootstrap replicates, default 50) and
`HARDEN_STRICT` (`1` makes the last cell fail on any unexpected error).

In [1]:
import faulthandler
import json
import os
import sys
import time
import traceback
import warnings
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd

for _c in (Path.cwd(), *Path.cwd().parents):
    if (_c / "hardening" / "datasets" / "registry.py").exists():
        REPO = _c
        break
sys.path.insert(0, str(REPO / "hardening" / "datasets"))
# Inside a Jupyter kernel sys.stderr has no file descriptor; give the watchdog a real file.
_FH = open(REPO / "hardening" / "logs" / "faulthandler_compas.log", "w")
faulthandler.dump_traceback_later(600, repeat=True, file=_FH)
import registry  # noqa: E402

import isitfair  # noqa: E402
from isitfair import FairnessAudit, GroupRecalibration, WassersteinPostprocessing, report_to_pdf  # noqa: E402

NAME = "compas"
B = int(os.environ.get("HARDEN_B", "50"))
STRICT = os.environ.get("HARDEN_STRICT", "0") == "1"
# Bootstrap workers for discrimination/calibration; results do not depend on it.
NJOBS = int(os.environ["HARDEN_NJOBS"]) if os.environ.get("HARDEN_NJOBS") else None
SEED = 42
OUT = REPO / "hardening" / "notebooks" / "results"
REPORTS = REPO / "hardening" / "reports"
OUT.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
META = registry.meta(NAME)
THRESHOLD = META["threshold"]
RECORD = {"dataset": NAME, "B": B, "isitfair": isitfair.__version__,
          "isitfair_file": isitfair.__file__, "n_jobs": NJOBS, "stages": {}, "variants": {}}
print(f"isitfair {isitfair.__version__} from {isitfair.__file__}")
print(f"{NAME}: B={B} threshold={THRESHOLD} strict={STRICT} n_jobs={NJOBS}")


@contextmanager
def stage(name, expect_error=False):
    """Time a stage; record its warnings and any exception, then carry on."""
    entry = {"seconds": None, "warnings": [], "error": None, "expected_error": expect_error}
    RECORD["stages"][name] = entry
    t0 = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        try:
            yield entry
        except Exception as exc:  # recorded, not raised, so later stages still run
            entry["error"] = f"{type(exc).__name__}: {exc}"
            entry["traceback"] = traceback.format_exc()
            print(f"!! {name} raised {entry['error']}")
    entry["seconds"] = round(time.perf_counter() - t0, 3)
    seen = {}
    for w in caught:
        key = f"{w.category.__name__}: {str(w.message)[:300]}"
        seen[key] = seen.get(key, 0) + 1
    entry["warnings"] = [f"{k} (x{v})" for k, v in seen.items()]
    print(f"[{name}] {entry['seconds']:.2f} s, {len(caught)} warning(s)"
          + (f", ERROR {entry['error']}" if entry["error"] else ""))
    for k in entry["warnings"][:8]:
        print("   ", k)
    (OUT / f"{NAME}_B{B}.json").write_text(json.dumps(RECORD, indent=2, default=str))

isitfair 0.4.0 from C:\Users\hugoa\Desktop\isitfair\src\isitfair\__init__.py
compas: B=50 threshold=0.5 strict=True n_jobs=-1


## 1. Load and validate inputs

In [2]:
with stage("load_validate"):
    df = registry.load(NAME)
    ATTRS = registry.ATTRS[NAME]
    checks = {
        "n": len(df),
        "index_is_range": isinstance(df.index, pd.RangeIndex),
        "y_true_dtype": str(df["y_true"].dtype),
        "y_true_values": sorted(df["y_true"].dropna().unique().tolist()),
        "y_true_missing": int(df["y_true"].isna().sum()),
        "y_proba_range": [float(df["y_proba_lr"].min()), float(df["y_proba_lr"].max())],
        "y_proba_missing": int(df["y_proba_lr"].isna().sum()),
        "attr_dtypes": {a: str(df[a].dtype) for a in ATTRS},
        "attr_missing": {a: int(df[a].isna().sum()) for a in ATTRS},
        "attr_levels": {a: int(df[a].nunique()) for a in ATTRS},
    }
    RECORD["checks"] = checks
    print(json.dumps(checks, indent=1))

{
 "n": 1852,
 "index_is_range": true,
 "y_true_dtype": "int64",
 "y_true_values": [
  0,
  1
 ],
 "y_true_missing": 0,
 "y_proba_range": [
  0.0670734904062096,
  0.9977325800185624
 ],
 "y_proba_missing": 0,
 "attr_dtypes": {
  "race": "object",
  "sex": "object",
  "age_cat": "object"
 },
 "attr_missing": {
  "race": 0,
  "sex": 0,
  "age_cat": 0
 },
 "attr_levels": {
  "race": 6,
  "sex": 2,
  "age_cat": 3
 }
}
[load_validate] 0.01 s, 0 warning(s)


A missing subgroup label must be rejected with an actionable message. If any
attribute has missing labels, try the raw column first (recorded), then
recode to an explicit `"Unknown"` category for the rest of the notebook.

In [3]:
missing_attrs = [a for a in ATTRS if df[a].isna().any()]
if missing_attrs:
    with stage("raw_missing_labels", expect_error=True):
        FairnessAudit(y_true=df["y_true"], y_proba=df["y_proba_lr"],
                      sensitive_features={a: df[a] for a in ATTRS},
                      threshold=THRESHOLD, n_bootstrap=B, random_state=SEED, n_jobs=NJOBS)
    for a in missing_attrs:
        df[a] = df[a].fillna("Unknown")
print("recoded to 'Unknown':", missing_attrs or "none")

recoded to 'Unknown': none


In [4]:
y = df["y_true"].to_numpy(dtype=float)
p = df["y_proba_lr"].to_numpy(dtype=float)
marginals = {a: df[a].to_numpy(dtype=object) for a in ATTRS}
INTERSECTION = tuple(ATTRS[:2]) if len(ATTRS) >= 2 else None
sf = dict(marginals)
if INTERSECTION:
    sf[INTERSECTION] = None
print("intersection:", INTERSECTION)

intersection: ('race', 'sex')


## 2. Construct the audit

In [5]:
with stage("construct"):
    audit = FairnessAudit(y_true=y, y_proba=p, sensitive_features=sf, threshold=THRESHOLD,
                          n_bootstrap=B, random_state=SEED, n_jobs=NJOBS)
    print(audit)

FairnessAudit(n=1852, events=843, threshold=0.5, n_bootstrap=50, random_state=42, attributes: race (6), sex (2), age_cat (3), race × sex (10))
[construct] 0.01 s, 0 warning(s)


## 3. Discrimination

In [6]:
with stage("discrimination"):
    disc = audit.discrimination()
    print(disc[disc.subgroup != "GAP"][["attribute", "subgroup", "n", "n_events", "auroc",
                                         "auroc_ci_low", "auroc_ci_high"]].to_string(index=False))

 attribute                  subgroup      n  n_events    auroc  auroc_ci_low  auroc_ci_high
      race                   Overall 1852.0     843.0 0.732259      0.712080       0.755626
      race          African-American  956.0     511.0 0.741529      0.720202       0.776369
      race                     Asian   11.0       3.0 0.958333           NaN            NaN
      race                 Caucasian  645.0     240.0 0.677016      0.627768       0.718815
      race                  Hispanic  141.0      60.0 0.697428      0.627597       0.776446
      race           Native American    2.0       0.0      NaN           NaN            NaN
      race                     Other   97.0      29.0 0.771805      0.664868       0.862445
       sex                   Overall 1852.0     843.0 0.732259      0.712080       0.755626
       sex                    Female  360.0     127.0 0.700990      0.647615       0.751514
       sex                      Male 1492.0     716.0 0.731646      0.707570    

## 4. Calibration

In [7]:
with stage("calibration"):
    cal = audit.calibration()
    print(cal[cal.subgroup != "GAP"][["attribute", "subgroup", "n", "calibration_intercept",
                                       "calibration_slope", "brier", "ece", "ici"]].to_string(index=False))

 attribute                  subgroup      n  calibration_intercept  calibration_slope    brier      ece      ici
      race                   Overall 1852.0               0.009677           1.027282 0.209219 0.041049 0.026934
      race          African-American  956.0               0.095103           1.046432 0.209822 0.074896 0.042466
      race                     Asian   11.0              -0.061870           7.563164 0.116648 0.298700 0.199071
      race                 Caucasian  645.0              -0.049542           0.842944 0.213233 0.029144 0.016867
      race                  Hispanic  141.0               0.075151           0.886114 0.217166 0.110006 0.044219
      race           Native American    2.0                    NaN                NaN 0.230601 0.479962      NaN
      race                     Other   97.0              -0.520760           2.051900 0.175086 0.149337 0.123532
       sex                   Overall 1852.0               0.009677           1.027282 0.209219 0

## 5. Calibration curves

In [8]:
with stage("calibration_curves"):
    curves = audit.calibration_curves()
    print(curves.shape)

(2500, 7)
[calibration_curves] 11.77 s, 0 warning(s)


## 6. ROC curves

In [9]:
with stage("roc_curves"):
    roc = audit.roc_curves()
    print(roc.shape)

(2500, 7)
[roc_curves] 0.96 s, 0 warning(s)


## 7. Decision curve

In [10]:
with stage("decision_curve"):
    dc = audit.decision_curve()
    print(dc[dc.subgroup == "GAP"][["attribute", "gap_value", "gap_ci_low", "gap_ci_high"]].to_string(index=False))

 attribute  gap_value  gap_ci_low  gap_ci_high
      race   0.944909         NaN          NaN
       sex   0.185060    0.139049     0.248390
   age_cat   0.306179    0.216465     0.374698
race × sex   0.975703         NaN          NaN
[decision_curve] 2.25 s, 0 warning(s)


## 8. Estimability

In [11]:
with stage("estimability"):
    est = audit.estimability()
    print(f"{int(est.not_estimable_final.sum())} of {len(est)} cell-metrics not reportable")

84 of 303 cell-metrics not reportable
[estimability] 0.03 s, 0 warning(s)


## 9. Hosmer-Lemeshow

In [12]:
with stage("hosmer_lemeshow"):
    hl = audit.hosmer_lemeshow()
    print(hl[hl.subgroup != "GAP"][["attribute", "subgroup", "n", "hl_statistic", "hl_df",
                                     "hl_p_value", "n_bins_used"]].to_string(index=False))

 attribute                  subgroup      n  hl_statistic  hl_df  hl_p_value  n_bins_used
      race                   Overall 1852.0     19.235491   10.0    0.037372         10.0
      race          African-American  956.0     36.056401   10.0    0.000082         10.0
      race                     Asian   11.0      5.865420   10.0    0.826440         10.0
      race                 Caucasian  645.0      5.363050   10.0    0.865645         10.0
      race                  Hispanic  141.0     10.993959   10.0    0.357989         10.0
      race           Native American    2.0           NaN    NaN         NaN          2.0
      race                     Other   97.0     14.319202   10.0    0.158923         10.0
       sex                   Overall 1852.0     19.235491   10.0    0.037372         10.0
       sex                    Female  360.0     25.366341   10.0    0.004693         10.0
       sex                      Male 1492.0     14.975687   10.0    0.132951         10.0
   age_cat

## 10. Cell diagnostics for one intersection

In [13]:
with stage("cell_diagnostics"):
    if INTERSECTION:
        cells = audit.cell_diagnostics()
        print(cells.to_string(index=False))
    else:
        print("single attribute; no intersection to diagnose")

 attribute                  subgroup   n  n_events  shrunk  marginal_rate  cell_rate  shrinkage_factor
race × sex African-American / Female 174        68   False       0.534519   0.390805          0.617582
race × sex   African-American / Male 782       443   False       0.534519   0.566496          0.264346
race × sex              Asian / Male  11         3    True       0.272727   0.272727          0.962329
race × sex        Caucasian / Female 139        46    True       0.372093   0.330935          0.669048
race × sex          Caucasian / Male 506       194   False       0.372093   0.383399          0.357052
race × sex         Hispanic / Female  24        11    True       0.425532   0.458333          0.921311
race × sex           Hispanic / Male 117        49    True       0.425532   0.418803          0.706030
race × sex    Native American / Male   2         0    True       0.000000   0.000000          0.992933
race × sex            Other / Female  23         2    True       0.298969

## 11. Recalibration ladder on the attribute with the largest intercept gap

In [14]:
gaps = cal[(cal.subgroup == "GAP") & (cal.gap_metric == "max_intercept_gap")
           & cal.attribute.isin(ATTRS)]
RANKED = [str(a) for a in gaps.sort_values("gap_value", ascending=False).attribute]


def zero_event_levels(attr):
    ev = pd.Series(y).groupby(marginals[attr]).sum()
    return [str(k) for k, v in ev.items() if v == 0]


LADDER_BY = RANKED[0]
ZERO = zero_event_levels(LADDER_BY)
print("largest intercept gap:", LADDER_BY, "| zero-event levels:", ZERO or "none")
# A level without events cannot be recalibrated; isitfair refuses (by design).
with stage("recalibration_ladder", expect_error=bool(ZERO)):
    ladder = audit.recalibration_ladder(by=LADDER_BY)
    print(ladder.to_string(index=False)[:3000])

largest intercept gap: race | zero-event levels: ['Native American']
!! recalibration_ladder raised ValueError: stratum 'Native American' of 'race' has zero events in the whole audit, so no fold can fit a group-specific calibrator for it; fewer folds will not help. Merge it into another level or drop those rows.
[recalibration_ladder] 0.01 s, 0 warning(s), ERROR ValueError: stratum 'Native American' of 'race' has zero events in the whole audit, so no fold can fit a group-specific calibrator for it; fewer folds will not help. Merge it into another level or drop those rows.


In [15]:
FALLBACK = next((a for a in RANKED[1:] if not zero_event_levels(a)), None) if ZERO else None
if FALLBACK:
    with stage("recalibration_ladder_fallback"):
        ladder = audit.recalibration_ladder(by=FALLBACK)
        print("by", FALLBACK)
        print(ladder.to_string(index=False)[:3000])
else:
    print("no fallback needed" if not ZERO else "no attribute without zero-event levels")

by age_cat
          rung attribute        subgroup      n  n_events  calibration_intercept  intercept_ci_low  intercept_ci_high  calibration_slope  slope_ci_low  slope_ci_high    brier  brier_ci_low  brier_ci_high      ece      ici  net_benefit        gap_metric  gap_value  event_floor below_floor not_estimable_final
          none   age_cat         Overall 1852.0     843.0               0.009677         -0.089083           0.094173           1.027282      0.892957       1.161157 0.209219      0.203267       0.216251 0.041049 0.026934     0.139849               NaN        NaN         25.0       False               False
          none   age_cat         25 - 45 1027.0     481.0              -0.091520         -0.206696           0.024812           1.008438      0.831588       1.195697 0.213291      0.204959       0.221658 0.055819 0.032443     0.152872               NaN        NaN         25.0       False               False
          none   age_cat Greater than 45  403.0     124.0     

## 12. Mitigation on a held-out calibration split

support2 uses the committed calibration partition. Every other dataset splits
its audited test half 50/50 (stratified on the outcome, `random_state=42`):
the calibrators are fitted on one half and the before/after audit runs on the
other, so no row is scored by a calibrator fitted on it.

In [16]:
with stage("mitigation_split"):
    if NAME == "support2":
        cal_df = pd.read_csv(registry.PRED / "support2_calib.csv", dtype={a: object for a in ATTRS})
        eval_df = df
    else:
        from sklearn.model_selection import train_test_split
        i_cal, i_eval = train_test_split(np.arange(len(df)), test_size=0.5, random_state=SEED,
                                         stratify=df["y_true"])
        cal_df, eval_df = df.iloc[i_cal].reset_index(drop=True), df.iloc[i_eval].reset_index(drop=True)
    for a in ATTRS:
        cal_df[a] = cal_df[a].fillna("Unknown")
    calib_data = (cal_df["y_true"].to_numpy(float), cal_df["y_proba_lr"].to_numpy(float),
                  {a: cal_df[a].to_numpy(object) for a in ATTRS})
    sf_eval = {a: eval_df[a].to_numpy(object) for a in ATTRS}
    if INTERSECTION:
        sf_eval[INTERSECTION] = None
    audit_eval = audit if NAME == "support2" else FairnessAudit(
        y_true=eval_df["y_true"].to_numpy(float), y_proba=eval_df["y_proba_lr"].to_numpy(float),
        sensitive_features=sf_eval, threshold=THRESHOLD, n_bootstrap=B, random_state=SEED, n_jobs=NJOBS)
    print(f"calibration n={len(cal_df)}, evaluation n={len(eval_df)}")

calibration n=926, evaluation n=926
[mitigation_split] 0.01 s, 0 warning(s)


In [17]:
with stage("mitigate_group_recalibration"):
    mitigated_gr = audit_eval.mitigate(method=GroupRecalibration(method="isotonic"),
                                       calibration_data=calib_data)
    a0 = ATTRS[0]
    keep = ["subgroup", "calibration_intercept", "calibration_slope", "brier", "ece"]
    before, after = audit_eval.calibration(), mitigated_gr[a0].calibration()
    print("BEFORE", a0); print(before[before.attribute == a0][keep].to_string(index=False))
    print("AFTER (group isotonic)"); print(after[after.attribute == a0][keep].to_string(index=False))

BEFORE race
        subgroup  calibration_intercept  calibration_slope    brier      ece
         Overall              -0.001243           0.960563 0.214168 0.023138
African-American               0.085034           0.925654 0.216957 0.061299
           Asian               0.445948          24.417333 0.147097 0.369502
       Caucasian              -0.080470           0.778603 0.216235 0.079403
        Hispanic               0.275292           1.549092 0.201396 0.121350
           Other              -0.659676           1.100849 0.197010 0.191399
             GAP                    NaN                NaN      NaN      NaN
             GAP                    NaN                NaN      NaN      NaN
AFTER (group isotonic)
        subgroup  calibration_intercept  calibration_slope    brier      ece
         Overall              -0.030786           0.081434 0.224863 0.080370
African-American              -0.045822           0.313613 0.219964 0.066759
           Asian               0.950288  

In [18]:
with stage("mitigate_wasserstein"):
    mitigated_w = audit_eval.mitigate(method=WassersteinPostprocessing(random_state=SEED),
                                      calibration_data=calib_data)
    d_after = mitigated_w[a0].discrimination()
    print(d_after[(d_after.attribute == a0) & (d_after.subgroup == "GAP")][
        ["gap_metric", "gap_value", "gap_ci_low", "gap_ci_high"]].to_string(index=False))

                  gap_metric  gap_value  gap_ci_low  gap_ci_high
equal_opportunity_difference   0.600000         NaN          NaN
         predictive_equality   0.811659         NaN          NaN
          equalized_odds_gap   0.811659         NaN          NaN
          statistical_parity   0.660714         NaN          NaN
[mitigate_wasserstein] 0.88 s, 0 warning(s)


## 13. Report: HTML, then PDF

In [19]:
REPORT = REPORTS / f"{NAME}_B{B}.html"
with stage("report_html"):
    audit_eval.report(out=REPORT, title=f"{NAME}: hardening audit (B={B})",
                      metadata={"Dataset": NAME, "Model": "LogisticRegression (registry)",
                                "Threshold": str(THRESHOLD), "Bootstrap": f"B={B}, random_state={SEED}"},
                      mitigation_audits=mitigated_gr)
    print(REPORT, REPORT.stat().st_size, "bytes")

C:\Users\hugoa\Desktop\isitfair\hardening\reports\compas_B50.html 2053947 bytes
[report_html] 25.26 s, 0 warning(s)


In [20]:
with stage("report_pdf"):
    pdf_path = report_to_pdf(REPORT)
    print(pdf_path, pdf_path.stat().st_size, "bytes")

C:\Users\hugoa\Desktop\isitfair\hardening\reports\compas_B50.pdf 584942 bytes
[report_pdf] 12.92 s, 0 warning(s)


## 14. Input-type variants

The same audit (B = 20 to keep this cheap) with inputs passed in the forms
real pipelines produce. Each variant is classified as `raises`, `warns`,
`ok-identical` (results equal the plain-numpy baseline exactly), or
`ok-DIFFERENT` (runs silently but gives different numbers: a silent
misbehaviour unless the input itself changed, as in the ties variant).

In [21]:
VB = 20


def run_variant(name, y_in, p_in, sf_in, compare=True, ref=None):
    out = {"status": None, "warnings": [], "error": None}
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        try:
            a = FairnessAudit(y_true=y_in, y_proba=p_in, sensitive_features=sf_in,
                              threshold=THRESHOLD, n_bootstrap=VB, random_state=SEED, n_jobs=NJOBS)
            frames = {"discrimination": a.discrimination(), "calibration": a.calibration(),
                      "hosmer_lemeshow": a.hosmer_lemeshow(), "decision_curve": a.decision_curve()}
        except Exception as exc:
            out["error"] = f"{type(exc).__name__}: {exc}"
            frames = None
    out["warnings"] = sorted({f"{w.category.__name__}: {str(w.message)[:200]}" for w in caught})
    if frames is None:
        out["status"] = "raises"
    else:
        num_bad = {k: int(np.isinf(f.select_dtypes("number").to_numpy(float)).sum()) for k, f in frames.items()}
        out["inf_values"] = num_bad
        if ref is not None and compare:
            same = all(frames[k].equals(ref[k]) for k in frames)
            out["identical_to_baseline"] = same
            out["status"] = ("warns, " if out["warnings"] else "") + ("ok-identical" if same else "ok-DIFFERENT")
            if not same:
                diffs = {}
                for k in frames:
                    fa, fb = frames[k], ref[k]
                    if fa.shape == fb.shape:
                        na = fa.select_dtypes("number").to_numpy(float)
                        nb = fb.select_dtypes("number").to_numpy(float)
                        diffs[k] = float(np.nanmax(np.abs(na - nb))) if na.size else 0.0
                    else:
                        diffs[k] = f"shape {fa.shape} vs {fb.shape}"
                out["max_abs_diff"] = diffs
        else:
            out["status"] = ("warns, " if out["warnings"] else "") + "ok"
    RECORD["variants"][name] = out
    print(f"{name:<28} {out['status']}" + (f"  {out['error']}" if out["error"] else ""))
    for w in out["warnings"][:5]:
        print("     ", w)
    (OUT / f"{NAME}_B{B}.json").write_text(json.dumps(RECORD, indent=2, default=str))
    return frames


with stage("variants"):
    base = run_variant("baseline_numpy", y, p, sf, compare=False)
    s_attr = {a: pd.Series(v, dtype="category") for a, v in marginals.items()}
    run_variant("category_attributes", y, p, {**s_attr, **({INTERSECTION: None} if INTERSECTION else {})}, ref=base)
    run_variant("Int64_outcome", pd.Series(y.astype(int), dtype="Int64"), p, sf, ref=base)
    run_variant("bool_outcome", y.astype(bool), p, sf, ref=base)
    run_variant("bool_outcome_series", pd.Series(y.astype(bool)), p, sf, ref=base)
    perm = np.random.default_rng(SEED).permutation(len(y))
    idx = pd.Index(perm + 10_000)
    run_variant("shuffled_index_consistent",
                pd.Series(y, index=idx), pd.Series(p, index=idx),
                {**{a: pd.Series(v, index=idx) for a, v in marginals.items()},
                 **({INTERSECTION: None} if INTERSECTION else {})}, ref=base)
    # y sorted/shuffled with its labels, p left in the original order: the
    # classic silent misalignment, which isitfair is documented to reject.
    run_variant("shuffled_index_misaligned",
                pd.Series(y).iloc[perm], pd.Series(p), sf, ref=base)
    p_ties = np.round(p, 2)
    k = max(2, len(p) // 50)  # ~2% of rows exactly 0 or 1, never all of them
    pos = np.random.default_rng(SEED).choice(len(p), size=k, replace=False)
    p_ties[pos[: k // 2]] = 0.0
    p_ties[pos[k // 2 :]] = 1.0
    run_variant("ties_and_exact_0_1", y, p_ties, sf, compare=False)

baseline_numpy               ok


category_attributes          ok-identical


Int64_outcome                ok-identical


bool_outcome                 ok-identical


bool_outcome_series          ok-identical


shuffled_index_consistent    ok-identical
shuffled_index_misaligned    raises  ValueError: `y_proba` and `y_true` are pandas objects with the same index labels in a different order. FairnessAudit aligns inputs by position, not by index, so the rows would be mismatched. Reorder them consistently, e.g. `s.loc[other.index]`, or pass `.to_numpy()` arrays built from one DataFrame.


ties_and_exact_0_1           ok
[variants] 24.55 s, 0 warning(s)


## 15. Summary

In [22]:
unexpected = {k: v["error"] for k, v in RECORD["stages"].items()
              if v["error"] and not v["expected_error"]}
RECORD["unexpected_errors"] = unexpected
RECORD["total_seconds"] = round(sum(v["seconds"] or 0 for v in RECORD["stages"].values()), 2)
(OUT / f"{NAME}_B{B}.json").write_text(json.dumps(RECORD, indent=2, default=str))
print(pd.DataFrame({k: {"seconds": v["seconds"], "n_warnings": len(v["warnings"]),
                        "error": v["error"]} for k, v in RECORD["stages"].items()}).T.to_string())
print("unexpected errors:", unexpected or "none")
if STRICT:
    assert not unexpected, unexpected

                              seconds n_warnings                                                                                                                                                                                                                  error
load_validate                   0.008        0.0                                                                                                                                                                                                                    NaN
construct                       0.009        0.0                                                                                                                                                                                                                    NaN
discrimination                 11.293        0.0                                                                                                                                                                